In [64]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool,set_default_openai_client, set_default_openai_api
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [65]:
load_dotenv(override=True)
deepseek_client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
)

In [66]:
set_default_openai_client(deepseek_client)
set_default_openai_api("chat_completions")




In [67]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("alanmuchiri@gmail.com")  # Change to your verified sender
    to_email = To("muchirialan2@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


### Step 1: Agent workflow

In [68]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [69]:
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="deepseek-chat"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="deepseek-chat"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="deepseek-chat"
)

In [70]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC2 Compliance with AI-Powered Precision  

Dear [Recipient's First Name],  

I hope this message finds you well. My name is [Your Name], and I represent ComplAI, a leading provider of AI-driven solutions designed to simplify SOC2 compliance and audit preparation.  

In today’s fast-paced regulatory environment, ensuring compliance can be both time-consuming and resource-intensive. Many organizations struggle with manual processes, fragmented documentation, and the constant pressure of audit readiness. At ComplAI, we’ve developed a SaaS platform that leverages artificial intelligence to automate and streamline these challenges, reducing the burden on your team while enhancing accuracy and efficiency.  

Our solution offers:  
- **Automated Evidence Collection**: Seamlessly gather and organize compliance artifacts.  
- **Real-Time Gap Analysis**: Identify vulnerabilities and address them proactively.  
- **Audit-Ready Reporting**: Generate comprehensive, audit-

In [71]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Simplify SOC2 Compliance with AI-Powered Precision  

Dear [Recipient's First Name],  

I hope this message finds you well. At ComplAI, we specialize in transforming the complex and often daunting process of SOC2 compliance into a streamlined, efficient experience. Our AI-driven platform is designed to help organizations like yours achieve and maintain compliance with confidence, while significantly reducing the time and resources typically required for audit preparation.  

Given the increasing importance of data security and regulatory adherence, I wanted to introduce you to ComplAI. Our solution not only automates evidence collection and policy management but also provides real-time insights and actionable recommendations to ensure you’re always audit-ready.  

Would you be open to a brief 15-minute conversation next week to explore how ComplAI can support your compliance goals? I’d be happy to share a tailored overview of our platform and discuss how we’ve helped similar o

In [72]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
     model="deepseek-chat"
)

In [73]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
**Subject:** SOC2 compliance: Let’s make it less painful than a surprise audit 🚨  

Hey [First Name],  

I’m reaching out because I’ve heard whispers that SOC2 compliance can feel like trying to solve a Rubik’s Cube blindfolded—frustrating, time-consuming, and a little bit sweaty.  

But what if I told you there’s a way to turn that chaos into clarity?  

At ComplAI, we’ve built an AI-powered platform that automates SOC2 compliance and audit prep, so you can stop stressing over spreadsheets and start focusing on what really matters—like growing your business (or finally taking that lunch break).  

Our tool helps you:  
✅ **Automate evidence collection** – No more chasing down teammates for screenshots.  
✅ **Generate audit-ready reports** – Impress auditors with polished, precise documentation.  
✅ **Stay compliant 24/7** – Real-time monitoring means you’re always audit-ready.  

Think of us as your compliance co-pilot, minus the bad small talk.  

Would you be open 

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [75]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("alanmuchiri@gmail.com")  # Change to your verified sender
    to_email = To("muchirialan2@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [76]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x72f8eb5f0720>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x72f8f84cd620>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='

In [77]:


instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="deepseek-chat")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)